In [ ]:
!pip install -q feast==0.52.0 pandas==2.2.2 numpy==2.0.2 scikit-learn==1.6.1

### Imports

In [1]:
!uv add feast==0.52.0 pandas==2.2.2 numpy==2.0.2 scikit-learn==1.6.1

Resolved 154 packages in 1.45s
Prepared 27 packages in 4.39s
Uninstalled 2 packages in 334ms
Installed 46 packages in 1.12s
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + bigtree==1.3.0
 + click==8.3.1
 + cloudpickle==3.1.2
 + dask==2026.1.1
 + dill==0.3.9
 + fastapi==0.128.0
 + feast==0.52.0
 + fsspec==2026.1.0
 + greenlet==3.3.1
 + gunicorn==24.1.1
 + httptools==0.7.1
 + importlib-metadata==8.7.1
 + joblib==1.5.3
 + librt==0.7.8
 + locket==1.0.0
 + mmh3==5.2.0
 + mypy==1.19.1
 + mypy-extensions==1.1.0
 - numpy==2.2.6
 + numpy==2.0.2
 - pandas==2.3.3
 + pandas==2.2.2
 + partd==1.4.2
 + pathspec==1.0.3
 + protobuf==6.33.4
 + pyarrow==17.0.0
 + pydantic==2.10.6
 + pydantic-core==2.27.2
 + pyjwt==2.10.1
 + python-dotenv==1.2.1
 + scikit-learn==1.6.1
 + scipy==1.15.3
 + sqlalchemy==2.0.46
 + starlette==0.50.0
 + tabulate==0.9.0
 + tenacity==8.5.0
 + threadpoolctl==3.6.0
 + toml==0.10.2
 + toolz==1.1.0
 + tqdm==4.67.1
 + typeguard==4.4.4
 + uvicorn==0.34.0
 + uvicorn-worker==0.3.0
 +

In [2]:
import shutil, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone, timedelta

from feast import FeatureStore, Entity, Field, FileSource, FeatureView, FeatureService
from feast.types import Int64, Float32
from feast.value_type import ValueType

from sklearn.metrics import classification_report, roc_auc_score, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore", category=DeprecationWarning)

### Feast Repo Setup

In [17]:
REPO_DIR = Path("feast_telco_repo")
#wipes any previous repo and creates a fresh "feast_teleco_repo"
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
(REPO_DIR / "data").mkdir(parents=True, exist_ok=True)

#configures feast with offline store type:file (feast reads parquet/csv from disk)
#online store : sqlite DB for quick demo serving
(REPO_DIR / "feature_store.yaml").write_text("""
project: telco_churn
registry: "registry.db"
provider: local
offline_store:
  type: file
online_store:
  type: sqlite
  path: "online_store.db"
entity_key_serialization_version: 3
""".strip()+"\n")

180

### Fetch and Clean Dataset

In [18]:
URL = "https://raw.githubusercontent.com/aiplanethub/Datasets/master/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(URL).rename(columns={"customerID": "customer_id"}).copy() # fetches the dataset

# Convert numeric - fixes numerics
df["TotalCharges"]   = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["MonthlyCharges"] = pd.to_numeric(df["MonthlyCharges"], errors="coerce")
df["tenure"]         = pd.to_numeric(df["tenure"], errors="coerce")

# Drop NA criticals - 
df = df.dropna(subset=["customer_id", "tenure", "MonthlyCharges", "TotalCharges", "Churn"])

# builds binary label
df["label"] = (df["Churn"].astype(str).str.strip().str.lower() == "yes").astype(int)

In [19]:
categorical_features = [
    "gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService", "MultipleLines",
    "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport",
    "StreamingTV", "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod"
]
numerical_features = ["tenure", "MonthlyCharges", "TotalCharges"]
all_features = numerical_features + categorical_features

### Simulate Feature & Label Timestamps

In [20]:
HORIZON_DAYS = 30
rng_end = datetime.now(timezone.utc)
rng_start = rng_end - timedelta(days=150)  # room for horizon
feat_span_sec = (rng_end - timedelta(days=HORIZON_DAYS) - rng_start).total_seconds()

np.random.seed(42)
df["feature_ts"] = [rng_start + timedelta(seconds=int(np.random.rand() * feat_span_sec)) for _ in range(len(df))]
df["label_ts"]   = df["feature_ts"] + timedelta(days=HORIZON_DAYS)
df["created_at"] = df["feature_ts"] + timedelta(minutes=5)  # ingestion lag

### Encode Data for Feast Storage

In [21]:
df_feast = df.copy()
for col in categorical_features:
    le = LabelEncoder()
    df_feast[col] = le.fit_transform(df_feast[col].astype(str))
#int64 fields cant store raw strings,so we create encoded copy for storage,keeping original strings for model training later

### Save Feature + Label Datasets

In [28]:
features_path = REPO_DIR / "data" / "telco_features.parquet"#input features for feasts offline store
entity_label_path = REPO_DIR / "data" / "entity_labels.parquet" #labels + timestamps for ML training

df_feast[["customer_id", *all_features, "feature_ts", "created_at"]].rename(
    columns={"feature_ts": "event_timestamp"}
).to_parquet(features_path, index=False)

df[["customer_id", "label_ts", "label"]].rename(
    columns={"label_ts": "event_timestamp"}
).to_parquet(entity_label_path, index=False)

### Define Feast Entity, Source, and FeatureView

In [30]:
customer = Entity(
    name="customer_id",
    join_keys=["customer_id"],
    value_type=ValueType.STRING,
)#Entity Definition

source = FileSource(
    path="data/telco_features.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_at",  # guards against late/backfilled data
)#Source Definition

# Use Float32 for numeric features, Int64 for encoded categoricals
schema = [
    Field(name=col, dtype=(Float32 if col in numerical_features else Int64))
    for col in all_features
]#Schema Definition

customer_stats = FeatureView(
    name="customer_stats",
    entities=[customer],
    ttl=timedelta(days=365),
    schema=schema,
    source=source,
    online=True,
)#FeatureView Definition

store = FeatureStore(repo_path=str(REPO_DIR))
store.apply([customer, customer_stats]) #Apply to FeatureStore Object

### Retrieve Point-in-Time Correct Features

In [33]:
entity_df = pd.read_parquet(entity_label_path)
# print(f"entity df: {entity_df}")
training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[f"customer_stats:{f}" for f in all_features],
).to_df() # retrieve historical features 
# print(f"training df: {training_df}")
training_df = training_df.dropna(subset=all_features + ["label"])
training_df["label"] = training_df["label"].astype(int)

### Time-Based Split (Leakage Safe)

In [34]:
ts = training_df["event_timestamp"]
q_train = ts.quantile(0.70)#oldest -> train
q_val = ts.quantile(0.85) #next % -> validation

train_df = training_df[ts <= q_train].copy()
val_df = training_df[(ts > q_train) & (ts <= q_val)].copy()
test_df = training_df[ts > q_val].copy() # most recent % -> test 

X_tr, y_tr = train_df[all_features], train_df["label"]
X_va, y_va = val_df[all_features], val_df["label"]
X_te, y_te = test_df[all_features], test_df["label"]

### ML Preprocessing Pipeline

In [35]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

pipeline = Pipeline([
    ("pre", preprocessor),
    ("clf", LogisticRegression(max_iter=5000, class_weight="balanced"))
])

In [36]:
pipeline.fit(X_tr, y_tr)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['tenure', 'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['gender', 'SeniorCitizen',
                                                   'Partner', 'Dependents',
                                                   'PhoneService',
                                                   'MultipleLines',
                                                   'InternetService',
                                                   'OnlineSecurity',
                                                   'OnlineBackup',
                                                   'DeviceProtection',
                                                   'TechSupport', 'StreamingTV',
                                                   'StreamingMovies',
                                                   'Contract',
                                                   'PaperlessBilling',
                                                   'PaymentMethod'])])),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=5000))])

### Threshold Tuning

In [37]:
y_prob_va = pipeline.predict_proba(X_va)[:, 1]
thresholds = np.linspace(0.0, 1.0, 201)
best_t, best_f1 = 0.5, -1.0

for t in thresholds:
    preds = (y_prob_va >= t).astype(int)
    f1 = f1_score(y_va, preds, zero_division=0)
    if f1 > best_f1:
        best_t, best_f1 = t, f1

In [38]:
y_prob_te = pipeline.predict_proba(X_te)[:, 1]
y_pred_te = (y_prob_te >= best_t).astype(int)
print(f"\nChosen threshold (from val): {best_t:.2f}")
print(f"Test ROC AUC:   {roc_auc_score(y_te, y_prob_te):.4f}")
print("\n=== Test Classification Report ===")
print(classification_report(y_te, y_pred_te, digits=4))


Chosen threshold (from val): 0.57
Test ROC AUC:   0.8293

=== Test Classification Report ===
              precision    recall  f1-score   support

           0     0.8939    0.7589    0.8209       788
           1     0.5078    0.7341    0.6003       267

    accuracy                         0.7526      1055
   macro avg     0.7008    0.7465    0.7106      1055
weighted avg     0.7962    0.7526    0.7650      1055



### Materialize to Online Store & Online Feature Lookup (Serving Simulation)

In [40]:
#materialization : process of loading feature data from an offline store into an online store, to simulate serving, we materialize features into online store and query them by entity keys 
store.materialize_incremental(end_date=datetime.now(timezone.utc))

svc = FeatureService(name="customer_stats_service", features=[customer_stats])
store.apply([svc])
#materialize features to online store and then define a feature service

sample_ids = training_df["customer_id"].drop_duplicates().sample(5, random_state=42).tolist()
online = store.get_online_features(
    features=svc,
    entity_rows=[{"customer_id": cid} for cid in sample_ids],
).to_dict() #fetch latest feature values from the online store 

print("\n=== Online features sample ===")
for i, cid in enumerate(sample_ids):
    row = {k: v[i] for k, v in online.items()}
    print(f"{cid}: {row}")

Materializing 1 feature views to 2026-01-26 15:03:57+00:00 into the sqlite online store.

customer_stats from 2026-01-26 15:02:27+00:00 to 2026-01-26 15:03:57+00:00:

=== Online features sample ===
2346-CZYIL: {'customer_id': '2346-CZYIL', 'PaperlessBilling': 0, 'Contract': 2, 'Dependents': 0, 'PhoneService': 1, 'MonthlyCharges': 20.350000381469727, 'InternetService': 2, 'tenure': 27.0, 'Partner': 0, 'OnlineBackup': 1, 'TotalCharges': 531.5999755859375, 'OnlineSecurity': 1, 'StreamingTV': 1, 'MultipleLines': 0, 'TechSupport': 1, 'SeniorCitizen': 0, 'PaymentMethod': 1, 'DeviceProtection': 1, 'StreamingMovies': 1, 'gender': 1}
6408-OTUBZ: {'customer_id': '6408-OTUBZ', 'PaperlessBilling': 1, 'Contract': 1, 'Dependents': 0, 'PhoneService': 1, 'MonthlyCharges': 104.55000305175781, 'InternetService': 1, 'tenure': 66.0, 'Partner': 0, 'OnlineBackup': 2, 'TotalCharges': 6779.0498046875, 'OnlineSecurity': 2, 'StreamingTV': 2, 'MultipleLines': 2, 'TechSupport': 2, 'SeniorCitizen': 0, 'PaymentMeth